# Demo how to use cryptography for secure caching
Demonstrates
- EncryptedCache
- This was refactored from 010_demo_encrypted_cache_1_nolib.ipynb

IMPORTANT: If your notebook should output any sensitive date, make certain to install nbstripout as a commit hook in git so that these secrets do not get archived in git

In [ ]:
%load_ext autoreload
%autoreload 2

from encrypted_cache import EncryptedCache, get_hashed_filename

In [ ]:
from datetime import datetime, timezone
import logging

# Configure logging for the demo — set to DEBUG to see all messages, INFO for operational only
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("encrypted_cache").setLevel(logging.INFO)


secrets = {
    "MyTestSecret": "MyTestSecretKey",
    "AnotherSecret": "AnotherSecretKey",
}

cache = EncryptedCache(secrets, salt=b"demo-crypto")

# Cache miss handling
cache_id = "api_response"

def fetch_data():
    return {"value": "some data that we want to cache"}

result = cache.execute_cached(
    "MyTestSecret",
    fetch_data,
    cache_id,
    ttl="5 days",
    validasof_datetime=datetime.now(timezone.utc),
    comment="daily refresh",
)
print(result)

In [ ]:
def callback_callee(d: datetime, name: str) -> str:
    return f"Hello {name} at {d.strftime('%Y-%m-%d')} - I will now retrieve your code"


def get_callback(aname, dnow):
    def innercallback():
        return callback_callee(dnow, aname)
    return innercallback


aname = "Ake"
dnow = datetime.now(timezone.utc)
callback = get_callback(aname, dnow)

callback_description = f"get_callback(aname={aname}, dnow={dnow.strftime("%Y-%m-%d")})"
hashed_id = get_hashed_filename(callback_description, prefix="cd-demo")

result_hashed = cache.execute_cached(
    "MyTestSecret",
    callback,
    hashed_id,
    ttl="5 days",
    validasof_datetime=dnow,
    comment="hashed filename demo",
)
print(result_hashed)